# 3. Fault 조기탐지 검증

## 요약

종료시간이나 최종농도를 사용하지 않고 발효 시작 후 50·100·150시간까지의 공정값만 사용했다. 배치 단위 교차검증에서 150시간 모델이 ROC-AUC 0.834, PR-AUC 0.564로 가장 좋았지만 Fault가 10개뿐이므로 탐색 결과이며 운영 모델로 확정할 수 없다. CSV는 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
batch_end = data.groupby('배치번호')['발효시간(h)'].max()
print(f'배치 {data["배치번호"].nunique()}개, Fault {(batch_end.index > 90).sum()}개')
print(f'최소 종료시간: {batch_end.min():.1f}시간')

배치 100개, Fault 10개
최소 종료시간: 167.0시간


### 판단

모든 배치가 150시간 이후에 종료되므로 세 관찰시점 모두 같은 100개 배치를 비교할 수 있다. 각 배치의 실제 종료시간 비율을 사용하지 않아 미래정보 누출을 피했다.

In [2]:
features = ['산누적량', 'DO평균', 'DO최솟값', 'pH표준편차', 'OUR평균', 'OUR음수비율', 'CO2평균', '기질기울기']


def make_early_features(horizon):
    rows = []
    for batch_number, batch in data.groupby('배치번호', sort=True):
        batch = batch.sort_values('발효시간(h)')
        observed = batch.loc[batch['발효시간(h)'] <= horizon]
        time = observed['발효시간(h)'].to_numpy()
        substrate = observed['기질농도(g/L)'].to_numpy()
        our = observed['산소소모율(g/min)'].to_numpy()
        rows.append({
            '배치번호': batch_number, 'Fault': int(batch_number > 90),
            '산누적량': np.trapezoid(observed['산투입유량(L/h)'], time),
            'DO평균': observed['용존산소(mg/L)'].mean(),
            'DO최솟값': observed['용존산소(mg/L)'].min(),
            'pH표준편차': observed['pH'].std(ddof=1),
            'OUR평균': our.mean(), 'OUR음수비율': np.mean(our < 0),
            'CO2평균': observed['배가스이산화탄소(%)'].mean(),
            '기질기울기': np.polyfit(time, substrate, 1)[0],
        })
    return pd.DataFrame(rows)


for horizon in [50, 100, 150]:
    preview = make_early_features(horizon)
    print(horizon, '시간:', preview.shape, '결측치', int(preview[features].isna().sum().sum()))

50 시간: (100, 10) 결측치 0
100 시간: (100, 10) 결측치 0
150 시간: (100, 10) 결측치 0


### 판단

최종농도·농도유지율·총수확량은 종료 후에만 알 수 있으므로 입력에서 제외했다. 각 시점의 8개 입력변수에는 결측치가 없다.

In [3]:
def fit_ridge_logistic(x, y, penalty=1.0, max_iterations=100):
    coefficients = np.zeros(x.shape[1])
    penalty_matrix = np.eye(x.shape[1])
    penalty_matrix[0, 0] = 0
    for _ in range(max_iterations):
        probability = 1 / (1 + np.exp(-np.clip(x @ coefficients, -30, 30)))
        weight = np.clip(probability * (1 - probability), 1e-8, None)
        gradient = x.T @ (y - probability) - penalty * penalty_matrix @ coefficients
        hessian = x.T @ (weight[:, None] * x) + penalty * penalty_matrix
        step = np.linalg.solve(hessian, gradient)
        coefficients = coefficients + step
        if np.max(np.abs(step)) < 1e-8:
            break
    return coefficients


def roc_auc(y_true, scores):
    positive_count = y_true.sum()
    negative_count = len(y_true) - positive_count
    ranks = stats.rankdata(scores)
    return (ranks[y_true == 1].sum() - positive_count * (positive_count + 1) / 2) / (positive_count * negative_count)


def average_precision(y_true, scores):
    order = np.argsort(-scores)
    ordered_y = y_true[order]
    precision = np.cumsum(ordered_y) / np.arange(1, len(y_true) + 1)
    return np.sum(precision * ordered_y) / ordered_y.sum()


rng = np.random.default_rng(42)
performance_rows = []
feature_tables = {}
for horizon in [50, 100, 150]:
    early = make_early_features(horizon)
    target = early['Fault'].to_numpy()
    predictions = np.zeros(len(early))
    for test_index in range(len(early)):
        train = np.arange(len(early)) != test_index
        train_values = early.loc[train, features]
        mean = train_values.mean().to_numpy()
        std = train_values.std(ddof=1).replace(0, 1).to_numpy()
        train_design = np.column_stack([np.ones(train.sum()), (train_values.to_numpy() - mean) / std])
        test_design = np.r_[1, (early.loc[test_index, features].to_numpy() - mean) / std]
        coefficients = fit_ridge_logistic(train_design, target[train])
        predictions[test_index] = 1 / (1 + np.exp(-np.clip(test_design @ coefficients, -30, 30)))
    auc = roc_auc(target, predictions)
    ap = average_precision(target, predictions)
    top_ten = np.argsort(-predictions)[:10]
    top_ten_recall = target[top_ten].sum() / target.sum()
    null_auc = np.array([roc_auc(rng.permutation(target), predictions) for _ in range(5000)])
    permutation_p = (1 + np.sum(null_auc >= auc)) / (len(null_auc) + 1)
    performance_rows.append({
        '관찰시간': horizon, 'ROC_AUC': auc, 'PR_AUC': ap, '무작위_PR기준': target.mean(),
        '상위10개_재현율': top_ten_recall, 'AUC_순열p': permutation_p,
    })

performance = pd.DataFrame(performance_rows)
display(performance.round(6))

early_150 = make_early_features(150)
mean_150 = early_150[features].mean()
std_150 = early_150[features].std(ddof=1).replace(0, 1)
design_150 = np.column_stack([np.ones(len(early_150)), (early_150[features] - mean_150) / std_150])
coef_150 = fit_ridge_logistic(design_150, early_150['Fault'].to_numpy())
display(pd.DataFrame({'변수': ['절편'] + features, '표준화계수': coef_150, '오즈비': np.exp(coef_150)}).round(6))

,관찰시간,ROC_AUC,PR_AUC,무작위_PR기준,상위10개_재현율,AUC_순열p
0,50,0.730000,0.250389,0.1,0.2,0.005999
1,100,0.606667,0.216728,0.1,0.2,0.142372
2,150,0.834444,0.564117,0.1,0.5,0.000200


,변수,표준화계수,오즈비
0,절편,-3.174286,0.041824
1,산누적량,0.512408,1.669306
2,DO평균,0.106361,1.112223
3,DO최솟값,0.487492,1.628228
4,pH표준편차,0.337123,1.400912
5,OUR평균,-0.037185,0.963498
6,OUR음수비율,0.911765,2.488710
7,CO2평균,0.076555,1.079561
8,기질기울기,0.916572,2.500704


### 최종 판단

- 50시간 모델은 ROC-AUC 0.730, PR-AUC 0.250으로 무작위 PR 기준 0.10보다는 높지만 상위 10개 배치에서 Fault 2개만 찾았다.
- 100시간 성능은 ROC-AUC 0.607, PR-AUC 0.217로 불안정했다. 시간이 늘면 항상 좋아지는 모델이 아니므로 배치별 운전 단계 차이와 변수 궤적을 추가로 확인해야 한다.
- 150시간 모델은 ROC-AUC 0.834, PR-AUC 0.564이고 상위 10개에서 Fault 5개를 찾았다. 세 시점 중 가장 유망하지만 완전한 탐지는 아니다.
- 150시간 전체자료 계수에서는 OUR 음수비율과 기질 기울기의 양의 계수가 컸다. 변수 간 상관과 작은 Fault 표본 때문에 원인이나 확정 임계값으로 해석하면 안 된다.
- 배치가 100개, Fault가 10개뿐이고 같은 시뮬레이션 데이터에서 평가했다. 운영 적용 전 별도 배치 검증, 고정 경보 임계값, 민감도·오경보율 검증이 필수다.